In [ ]:
import json

ENGINEER_MOCK_DATA_LOCATION = 'engineer_mock_data.json'
TASKS_MOCK_DATA_LOCATION = 'tasks_mock_data.json'

# read file mock data
def read_file(file_path, validation_func):
    with open(file_path, 'r') as file:
        read_data = json.load(file)
        if (validation_func(read_data)):
            return read_data
        else:
            return []

# validate tasks list and object keys
def validate_tasks(tasks_mock): 
    if not isinstance(tasks_mock, list):
        return False
    for item in tasks_mock:
        if not isinstance(item, dict):
            return False
        if list(item.keys()) != ['task_id', 'skill_required', 'due_date', 'priority', 'dependency_level', 'module_knowledge', 'task_type']:
            return False
    return True

# validate engineer list and object keys
def validate_engineer(engineer_mock):
    if not isinstance(engineer_mock, list):
        return False
    for item in engineer_mock:
        if not isinstance(item, dict):
            return False
        if list(item.keys()) != ['engineer_id', 'skills', 'collab_ability', 'availability_index', 'impact', 'module_exposure', 'efficiency']:
            return False
    return True

engineer_mock = read_file(ENGINEER_MOCK_DATA_LOCATION, validate_engineer)
tasks_mock = read_file(TASKS_MOCK_DATA_LOCATION, validate_tasks)

print(engineer_mock)
print(tasks_mock)

In [ ]:
ENGINEER_SKILL_KEY_NAME = 'skills'
TASK_SKILL_KEY_NAME = 'skill_required'

# sort engineers or tasks by skills 
def sort_items_by_skills(items_list, item_key_name):
    sorted_items_dict = {}
    for item in items_list:
        if not item[item_key_name] in sorted_items_dict:
            sorted_items_dict[item[item_key_name]] = []
        sorted_items_dict[item[item_key_name]].append(item)
    return sorted_items_dict

sorted_engineer_dict = sort_items_by_skills(engineer_mock, ENGINEER_SKILL_KEY_NAME)
sorted_tasks_dict = sort_items_by_skills(tasks_mock, TASK_SKILL_KEY_NAME)

print(sorted_engineer_dict)
print(sorted_tasks_dict)

In [ ]:
LEVELS_TO_NUMERICS = {
    'Critical': 4,
    'High': 3,
    'Medium': 2,
    'Low': 1
}
PLANNED_START_DATE = 1710144280 # 11 March 2024

def task_complexity_calculation(task):
    return (LEVELS_TO_NUMERICS[task['priority']] + LEVELS_TO_NUMERICS[task['dependency_level']])*task['module_knowledge']

def engineer_complexity_handling_calculation(engineer):
    return LEVELS_TO_NUMERICS[engineer['collab_ability']] * engineer['impact'] * engineer['module_exposure']

def task_time_calculation(task):
    return int(task['due_date']) - PLANNED_START_DATE

def engineer_time_efficiency_calculation(engineer):
    return engineer['availability_index'] * engineer['efficiency'] * 3600

# find time and complexity co-efficients for tasks and engineers
def item_time_complexity_list_generator(item_dict, time_function, complexity_function):
    for skill in list(item_dict.keys()):
        for item in item_dict[skill]:
            item['time_coef'] = time_function(item)
            item['complexity_coef'] = complexity_function(item)
    return item_dict

quantified_engineer_dict = item_time_complexity_list_generator(sorted_engineer_dict, engineer_time_efficiency_calculation, engineer_complexity_handling_calculation)
quantified_task_dict = item_time_complexity_list_generator(sorted_tasks_dict, task_time_calculation, task_complexity_calculation)

print(quantified_engineer_dict)
print(quantified_task_dict)

In [ ]:
import matplotlib.pyplot as plt

def plot_engineers_and_tasks_by_skill(engineer_dict, task_dict, skill):
    eng_items = engineer_dict.get(skill, [])
    task_items = task_dict.get(skill, [])

    # Extract coordinates
    eng_x = [e['time_coef'] for e in eng_items]
    eng_y = [e['complexity_coef'] for e in eng_items]

    task_x = [t['time_coef'] for t in task_items]
    task_y = [t['complexity_coef'] for t in task_items]

    plt.figure(figsize=(8, 6))

    # Engineers: Blue
    plt.scatter(eng_x, eng_y, color='red', label='Engineers', marker='^')

    # Tasks: Red
    plt.scatter(task_x, task_y, color='blue', label='Tasks')

    plt.xlabel('Time Coefficient')
    plt.ylabel('Complexity Coefficient')
    plt.title(f"Engineers vs Tasks for Skill: {skill}")
    plt.legend()
    plt.grid(True)
    plt.show()

# plotting the engineers and tasks on same graphs for visualisation
def plot_engineers_and_tasks(quantified_engineer_dict, quantified_task_dict):
    for skill in list(quantified_task_dict.keys()):
        plot_engineers_and_tasks_by_skill(quantified_engineer_dict, quantified_task_dict, skill)

plot_engineers_and_tasks(quantified_engineer_dict, quantified_task_dict)
